# Local Wakeword Bootstrap (Qwen + microWakeWord)

이 노트북은 로컬에서 `넙죽아` wakeword 모델을 만들기 위한 실행 허브입니다.
Qwen TTS(샘플 생성)와 microWakeWord(학습/평가)를 분리된 단계로 다룹니다.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)


def run(cmd: list[str], check: bool = False) -> int:
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, check=check)
    print("exit code:", proc.returncode)
    return proc.returncode


## 1) Environment Check

의존성 상태를 점검합니다. `strict=True`면 누락 시 실패 코드로 종료합니다.

In [ ]:
strict = False
cmd = [sys.executable, "-m", "nubjuk_wakeword.cli", "check-env"]
if strict:
    cmd.append("--strict")
run(cmd, check=False)


## 2) Korean Wakeword Synthesis (Qwen)

한국어 텍스트를 그대로 넣어 샘플을 생성하고, QC를 통과한 파일만 `generated_samples/`에 남깁니다.

In [ ]:
target_word = "넙죽아"
max_samples = 1000
batch_size = 8
run_synth = False

instructs = [
    "차분한 한국어 여성 목소리, 또렷한 발음",
    "부드러운 한국어 남성 목소리, 중간 속도",
    "밝고 경쾌한 톤, 자연스러운 한국어 발화",
    "조용한 환경에서 또렷하게 말하는 톤",
]

cmd = [
    sys.executable,
    "-m",
    "nubjuk_wakeword.cli",
    "synth",
    "--target-word",
    target_word,
    "--model-id",
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    "--language",
    "Korean",
    "--max-samples",
    str(max_samples),
    "--batch-size",
    str(batch_size),
    "--output-dir",
    "microWakeWord/notebooks/generated_samples",
]
for instr in instructs:
    cmd.extend(["--instruct", instr])

if run_synth:
    run(cmd, check=False)
else:
    cmd.append("--dry-run")
    run(cmd, check=False)


## 3) Train / Eval / Export Commands

아래 셀은 실행 커맨드만 구성합니다. 필요한 단계만 선택해 실행하세요.

In [ ]:
train_cmd = [sys.executable, "-m", "nubjuk_wakeword.cli", "train"]
eval_cmd = [
    sys.executable,
    "-m",
    "nubjuk_wakeword.cli",
    "eval",
    "--positives",
    "/Volumes/Gold-P31-SSD-2TB/wakeword/nubjuga",
]
export_cmd = [sys.executable, "-m", "nubjuk_wakeword.cli", "export"]

print("TRAIN:", " ".join(train_cmd))
print("EVAL :", " ".join(eval_cmd))
print("EXPORT:", " ".join(export_cmd))

run_train = False
run_eval = False
run_export = False

if run_train:
    run(train_cmd, check=False)
if run_eval:
    run(eval_cmd, check=False)
if run_export:
    run(export_cmd, check=False)


## 4) Inference Test (File + Realtime Mic)

학습된 모델로 파일/마이크 인퍼런스를 빠르게 확인합니다.

In [ ]:
model_path = "microWakeWord/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
file_input = "/Volumes/Gold-P31-SSD-2TB/wakeword/nubjuga"
file_threshold = 0.78
run_file_infer = False

file_cmd = [
    sys.executable,
    "scripts/06_try_model.py",
    "--model",
    model_path,
    "--input",
    file_input,
    "--threshold",
    str(file_threshold),
]
print("FILE INFER:", " ".join(file_cmd))

if run_file_infer:
    run(file_cmd, check=False)


In [ ]:
# basic_training_notebook.ipynb 의 realtime 셀과 동일한 흐름으로 실행
model_candidates = [
    PROJECT_ROOT / "microWakeWord/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite",
    PROJECT_ROOT / "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite",
]
model_path = next((p for p in model_candidates if p.exists()), None)
assert model_path is not None, "Model file not found. Run training first."

script_path = PROJECT_ROOT / "scripts/09_realtime_mic_test.py"
assert script_path.exists(), f"Realtime script not found: {script_path}"

selected_device = None  # 예: 4
cutoff = 0.78
preamp = 1.5
score_mode = "rolling_window"
rolling_window_ms = 1500
rolling_min_ms = 300
block_ms = 200
trigger_hold_blocks = 2
rearm_ratio = 0.6
rearm_hold_blocks = 2
rearm_threshold = None

run_list_devices = False
run_realtime_infer = False

list_devices_cmd = [sys.executable, str(script_path), "--list-devices"]
print("LIST DEVICES:", " ".join(list_devices_cmd))
if run_list_devices:
    run(list_devices_cmd, check=False)

realtime_cmd = [
    sys.executable,
    str(script_path),
    "--model",
    str(model_path),
    "--cutoff",
    str(cutoff),
    "--preamp",
    str(preamp),
    "--score-mode",
    str(score_mode),
    "--rolling-window-ms",
    str(rolling_window_ms),
    "--rolling-min-ms",
    str(rolling_min_ms),
    "--block-ms",
    str(block_ms),
    "--trigger-hold-blocks",
    str(trigger_hold_blocks),
    "--rearm-ratio",
    str(rearm_ratio),
    "--rearm-hold-blocks",
    str(rearm_hold_blocks),
]
if rearm_threshold is not None:
    realtime_cmd += ["--rearm-threshold", str(rearm_threshold)]
if selected_device is not None:
    realtime_cmd += ["--device", str(selected_device)]

print("REALTIME   :", " ".join(realtime_cmd))
if run_realtime_infer:
    run(realtime_cmd, check=False)
else:
    print("run_realtime_infer=False 이므로 실행하지 않았습니다. True로 바꿔 실행하세요.")
